# Character-CTC robustness LoRA — English + Korean

기존 adapter를 보존하고 언어별 새 adapter를 만듭니다.

- 각 언어 1,500 examples, 1 epoch
- clean Myanmar 1,050개 (70%)
- 실제 character CTC Myanmar 450개 (30%)
- English와 Korean은 서로 독립적으로 학습
- 마지막에 동일 held-out 300개로 Oracle과 Character CTC를 모두 평가

Kimi encoder와 character CTC는 동결되며, Qwen-token CTC 데이터는 사용하지 않습니다.


In [ ]:
!pip -q uninstall -y torchao
!pip -q install -U "transformers==4.57.3" "peft==0.17.1" accelerate safetensors pandas sacrebleu jiwer


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import gc,random,re,unicodedata,numpy as np,pandas as pd,torch
import torch.nn as nn,torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence,pack_padded_sequence,pad_packed_sequence
from torch.utils.data import Dataset,DataLoader
from tqdm.auto import tqdm
from transformers import AutoTokenizer,AutoModelForCausalLM
from peft import PeftModel
from sacrebleu.metrics import CHRF,BLEU

SEED=42;random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda');DTYPE=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
ROOT=Path('/content/drive/MyDrive/S2ST');SEM_DIR=ROOT/'baselines/bigru_prefix_semantic_alignment'
LORA_ROOT=ROOT/'baselines/qwen3_translation_lora';CHAR_EVAL=LORA_ROOT/'character_ctc_eval'
START={'en':LORA_ROOT/'clean_lora_best','ko':LORA_ROOT/'korean_clean_lora_epoch1'}
SAVE={'en':LORA_ROOT/'english_character_ctc_robustness_lora','ko':LORA_ROOT/'korean_character_ctc_robustness_lora'}
LLM_ID='Qwen/Qwen3-4B-Instruct-2507';TOTAL_PER_LANG=1500;NOISY_PER_LANG=450;CLEAN_PER_LANG=1050
BATCH_SIZE=2;ACCUM=8;LR=6e-6;MAX_LENGTH=384
assert TOTAL_PER_LANG==NOISY_PER_LANG+CLEAN_PER_LANG
for p in [SEM_DIR/'manifest.csv',SEM_DIR/'bigru_ctc_best.pt',CHAR_EVAL/'character_ctc_transcriptions.csv',*START.values()]:assert p.exists(),p
print(torch.cuda.get_device_name(),DTYPE)


Mounted at /content/drive


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


NVIDIA RTX PRO 6000 Blackwell Server Edition torch.bfloat16


## 1. Train split에서 source-balanced clean/noisy 표본 선택


In [ ]:
def clean_text(x):
    if x is None or (isinstance(x,float) and np.isnan(x)):return ''
    return ' '.join(unicodedata.normalize('NFC',str(x)).strip().split())
manifest=pd.read_csv(SEM_DIR/'manifest.csv',dtype={'sample_id':str})
for c in ['transcription_my','translation_en','translation_ko']:manifest[c]=manifest[c].map(clean_text)
train=manifest[manifest.split.eq('train')].copy()

def balanced_sample(frame,n,seed):
    parts=[];sources=sorted(frame.source.unique());base=n//len(sources)
    for i,s in enumerate(sources):parts.append(frame[frame.source.eq(s)].sample(base,random_state=seed+i))
    out=pd.concat(parts);remain=n-len(out)
    if remain:out=pd.concat([out,frame.drop(out.index).sample(remain,random_state=seed+99)])
    return out.sample(frac=1,random_state=seed).reset_index(drop=True)

noisy_meta=balanced_sample(train,NOISY_PER_LANG,SEED+10)
clean_pool=train[~train.sample_id.isin(noisy_meta.sample_id)]
clean_meta=balanced_sample(clean_pool,CLEAN_PER_LANG,SEED+20)
assert not set(noisy_meta.sample_id)&set(clean_meta.sample_id)
print('clean/noisy:',len(clean_meta),len(noisy_meta));display(pd.crosstab(noisy_meta.source,columns='noisy'))


clean/noisy: 1050 450


col_0,noisy
source,
fleurs,150
mig-burmese-audio-transcription,150
openslr,150


## 2. 450개 train 음성의 character CTC 출력 생성


In [ ]:
needed=set(noisy_meta.sample_id);feature_by_id={}
for path in [SEM_DIR/'kimi_features'/f'features-{i:05d}.pt' for i in range(5)]:
    assert path.exists(),path
    obj=torch.load(path,map_location='cpu',weights_only=False)
    for sid,feat in zip(map(str,obj['sample_ids']),obj['features']):
        if sid in needed:feature_by_id[sid]=feat.float()
    del obj;gc.collect()
assert set(feature_by_id)==needed

chars=sorted(set(''.join(manifest.transcription_my)));char_to_id={'<blank>':0,'<unk>':1}
for ch in chars:
    if ch not in char_to_id:char_to_id[ch]=len(char_to_id)
id_to_char={v:k for k,v in char_to_id.items()}
class BiGRU(nn.Module):
    def __init__(self):
        super().__init__();self.input=nn.Sequential(nn.LayerNorm(1280),nn.Linear(1280,256),nn.GELU(),nn.Dropout(.1));self.gru=nn.GRU(256,256,2,batch_first=True,bidirectional=True,dropout=.1)
    def forward(self,x,l):
        x=self.input(x);p=pack_padded_sequence(x,l.cpu(),batch_first=True,enforce_sorted=False);p,_=self.gru(p);return pad_packed_sequence(p,batch_first=True,total_length=x.shape[1])[0]
class CTC(nn.Module):
    def __init__(self):super().__init__();self.adapter=BiGRU();self.head=nn.Sequential(nn.LayerNorm(512),nn.Dropout(.1),nn.Linear(512,len(char_to_id)))
    def forward(self,x,l):return self.head(self.adapter(x,l))
class FD(Dataset):
    def __len__(self):return len(noisy_meta)
    def __getitem__(self,i):sid=noisy_meta.iloc[i].sample_id;return sid,feature_by_id[sid]
def fc(b):
    ids=[x[0] for x in b];f=[x[1].float() for x in b];return ids,pad_sequence(f,batch_first=True),torch.tensor([len(x) for x in f])
def decode(z):
    out=[];prev=None
    for x in z:
        x=int(x)
        if x!=prev and x not in (0,1):out.append(id_to_char[x])
        prev=x
    return clean_text(''.join(out))
ctc=CTC().to(DEVICE).float();ck=torch.load(SEM_DIR/'bigru_ctc_best.pt',map_location='cpu',weights_only=True);ctc.load_state_dict(ck['model']);ctc.eval()
pred={};loader=DataLoader(FD(),batch_size=16,collate_fn=fc,num_workers=0)
with torch.inference_mode():
    for ids,x,l in tqdm(loader,desc='train character CTC'):
        z=ctc(x.to(DEVICE,dtype=torch.float32),l).argmax(-1).cpu()
        for sid,row,n in zip(ids,z,l):pred[sid]=decode(row[:int(n)])
noisy_meta['ctc_text']=noisy_meta.sample_id.map(pred)
assert noisy_meta.ctc_text.str.len().gt(0).all() and not noisy_meta.ctc_text.str.contains('�',regex=False).any()
noisy_meta[['sample_id','source','transcription_my','ctc_text']].to_csv(LORA_ROOT/'character_ctc_robustness_train_inputs.csv',index=False)
del ctc,feature_by_id,loader;gc.collect();torch.cuda.empty_cache()


train character CTC:   0%|          | 0/29 [00:00<?, ?it/s]

## 3. 언어별 1,500개 학습 예제 구성


In [ ]:
def standard_prompt(text,lang):
    target='English' if lang=='en' else 'Korean'
    return f'Translate the following Myanmar text into {target}. Preserve its meaning and return only the translation.\n\nMyanmar:\n{text}'
def noisy_prompt(text,lang):
    target='English' if lang=='en' else 'Korean'
    restriction='Use English characters only.' if lang=='en' else 'Do not use Myanmar or Thai script.'
    return (f'Translate the following noisy Myanmar ASR text into {target}.\n'
            'Infer minor recognition errors from context without inventing facts.\n'
            f'Return only one {target} translation. {restriction}\n\nMyanmar:\n{text}')
def make_items(lang):
    ref=f'translation_{lang}';items=[]
    for r in clean_meta.itertuples():items.append({'sample_id':r.sample_id,'source':r.source,'kind':'clean','source_text':r.transcription_my,'reference':getattr(r,ref),'lang':lang})
    for r in noisy_meta.itertuples():items.append({'sample_id':r.sample_id,'source':r.source,'kind':'noisy','source_text':r.ctc_text,'reference':getattr(r,ref),'lang':lang})
    random.Random(SEED+(0 if lang=='en' else 1)).shuffle(items);return items
train_items={lang:make_items(lang) for lang in ['en','ko']}
for lang in train_items:
    f=pd.DataFrame(train_items[lang]);assert len(f)==1500;display(pd.crosstab(f.source,f.kind,margins=True))


kind,clean,noisy,All
source,,,
fleurs,350,150,500
mig-burmese-audio-transcription,350,150,500
openslr,350,150,500
All,1050,450,1500


kind,clean,noisy,All
source,,,
fleurs,350,150,500
mig-burmese-audio-transcription,350,150,500
openslr,350,150,500
All,1050,450,1500


## 4. English와 Korean adapter를 각각 1 epoch 학습


In [ ]:
def train_language(lang):
    tokenizer=AutoTokenizer.from_pretrained(START[lang],use_fast=True)
    if tokenizer.pad_token_id is None:tokenizer.pad_token=tokenizer.eos_token
    def encode(item):
        prompt=(standard_prompt if item['kind']=='clean' else noisy_prompt)(item['source_text'],lang)
        prefix=tokenizer.apply_chat_template([{'role':'user','content':prompt}],tokenize=False,add_generation_prompt=True)
        p=tokenizer(prefix,add_special_tokens=False).input_ids;t=tokenizer(item['reference'],add_special_tokens=False).input_ids+[tokenizer.eos_token_id]
        t=t[:max(1,MAX_LENGTH-len(p))];return torch.tensor((p+t)[:MAX_LENGTH]),torch.tensor(([-100]*len(p)+t)[:MAX_LENGTH])
    class DS(Dataset):
        def __len__(self):return len(train_items[lang])
        def __getitem__(self,i):return encode(train_items[lang][i])
    def collate(b):
        ids=pad_sequence([x[0] for x in b],batch_first=True,padding_value=tokenizer.pad_token_id);labels=pad_sequence([x[1] for x in b],batch_first=True,padding_value=-100)
        return ids,ids.ne(tokenizer.pad_token_id).long(),labels
    loader=DataLoader(DS(),batch_size=BATCH_SIZE,shuffle=True,collate_fn=collate,num_workers=0,pin_memory=True)
    base=AutoModelForCausalLM.from_pretrained(LLM_ID,dtype=DTYPE,device_map={'':'cuda:0'},attn_implementation='sdpa')
    model=PeftModel.from_pretrained(base,START[lang],is_trainable=True);model.config.use_cache=False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False});model.enable_input_require_grads()
    params=[p for p in model.parameters() if p.requires_grad];opt=torch.optim.AdamW(params,lr=LR,weight_decay=.01);model.train();opt.zero_grad(set_to_none=True);total=0
    for step,(ids,mask,labels) in enumerate(tqdm(loader,desc=f'{lang} CTC robustness 1/1'),1):
        out=model(input_ids=ids.to(DEVICE),attention_mask=mask.to(DEVICE),labels=labels.to(DEVICE),use_cache=False);loss=out.loss;(loss/ACCUM).backward();total+=loss.detach().float().item()
        if step%ACCUM==0 or step==len(loader):torch.nn.utils.clip_grad_norm_(params,1);opt.step();opt.zero_grad(set_to_none=True)
    SAVE[lang].mkdir(parents=True,exist_ok=True);model.save_pretrained(SAVE[lang]);tokenizer.save_pretrained(SAVE[lang]);print(lang,'mean loss',total/len(loader),'saved',SAVE[lang])
    del model,base,opt,loader;gc.collect();torch.cuda.empty_cache()
for lang in ['en','ko']:train_language(lang)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

...

en CTC robustness 1/1:   0%|          | 0/750 [00:00<?, ?it/s]

en mean loss nan saved /content/drive/MyDrive/S2ST/baselines/qwen3_translation_lora/english_character_ctc_robustness_lora


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

ko CTC robustness 1/1:   0%|          | 0/750 [00:00<?, ?it/s]

ko mean loss 0.9702023025155068 saved /content/drive/MyDrive/S2ST/baselines/qwen3_translation_lora/korean_character_ctc_robustness_lora


## 5. 동일 300개에서 Oracle + Character CTC 평가


In [ ]:
eval_ctc=pd.read_csv(CHAR_EVAL/'character_ctc_transcriptions.csv',dtype={'sample_id':str})
assert len(eval_ctc)==300
tokenizer=AutoTokenizer.from_pretrained(SAVE['en'],use_fast=True);tokenizer.padding_side='left'
if tokenizer.pad_token_id is None:tokenizer.pad_token=tokenizer.eos_token
base=AutoModelForCausalLM.from_pretrained(LLM_ID,dtype=DTYPE,device_map={'':'cuda:0'},attn_implementation='sdpa')
model=PeftModel.from_pretrained(base,SAVE['en'],adapter_name='english',is_trainable=False);model.load_adapter(SAVE['ko'],adapter_name='korean',is_trainable=False);model.eval()
@torch.inference_mode()
def generate(texts,lang,kind):
    model.set_adapter('english' if lang=='en' else 'korean');fn=standard_prompt if kind=='oracle' else noisy_prompt
    prompts=[tokenizer.apply_chat_template([{'role':'user','content':fn(x,lang)}],tokenize=False,add_generation_prompt=True) for x in texts]
    inp=tokenizer(prompts,return_tensors='pt',padding=True,truncation=True,max_length=768).to(DEVICE)
    out=model.generate(**inp,max_new_tokens=128,do_sample=False,repetition_penalty=1.05,eos_token_id=tokenizer.eos_token_id,pad_token_id=tokenizer.pad_token_id)
    return [x.strip() for x in tokenizer.batch_decode(out[:,inp.input_ids.shape[1]:],skip_special_tokens=True)]
rows=[]
for lang in ['en','ko']:
    for kind,col in [('oracle','transcription_my'),('ctc','character_ctc_text')]:
        answers=[]
        for start in tqdm(range(0,len(eval_ctc),8),desc=f'{lang} {kind} eval'):answers+=generate(eval_ctc[col].iloc[start:start+8].tolist(),lang,kind)
        for r,p in zip(eval_ctc.itertuples(),answers):rows.append({'sample_id':r.sample_id,'source':r.source,'language':lang,'condition':kind,'reference':getattr(r,f'translation_{lang}'),'prediction':p})
result=pd.DataFrame(rows);result.to_csv(LORA_ROOT/'character_ctc_robustness_predictions.csv',index=False)
chrf=CHRF(word_order=2);bleu=BLEU(effective_order=True);metrics=[]
for (condition,lang),g in result.groupby(['condition','language']):
    metrics.append({'condition':condition,'source':'all','language':lang,'samples':len(g),'chrf_pp':chrf.corpus_score(g.prediction.tolist(),[g.reference.tolist()]).score,'bleu':bleu.corpus_score(g.prediction.tolist(),[g.reference.tolist()]).score})
    for source,s in g.groupby('source'):metrics.append({'condition':condition,'source':source,'language':lang,'samples':len(s),'chrf_pp':chrf.corpus_score(s.prediction.tolist(),[s.reference.tolist()]).score,'bleu':bleu.corpus_score(s.prediction.tolist(),[s.reference.tolist()]).score})
metrics=pd.DataFrame(metrics);display(metrics);metrics.to_csv(LORA_ROOT/'character_ctc_robustness_metrics.csv',index=False)
baseline=pd.DataFrame([{'language':'en','old_ctc':26.609848,'old_oracle':44.980627},{'language':'ko','old_ctc':11.715379,'old_oracle':22.797998}])
now=metrics[metrics.source.eq('all')].pivot(index='language',columns='condition',values='chrf_pp').reset_index().rename(columns={'ctc':'new_ctc','oracle':'new_oracle'})
decision=baseline.merge(now,on='language');decision['ctc_gain']=decision.new_ctc-decision.old_ctc;decision['oracle_change']=decision.new_oracle-decision.old_oracle
decision['pass']=(decision.ctc_gain>=2)&(decision.oracle_change>=-1);display(decision);decision.to_csv(LORA_ROOT/'character_ctc_robustness_decision.csv',index=False)
display(result.groupby(['source','language','condition'],group_keys=False).head(2))


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

en oracle eval:   0%|          | 0/38 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


en ctc eval:   0%|          | 0/38 [00:00<?, ?it/s]

ko oracle eval:   0%|          | 0/38 [00:00<?, ?it/s]

ko ctc eval:   0%|          | 0/38 [00:00<?, ?it/s]

,condition,source,language,samples,chrf_pp,bleu
0,ctc,all,en,300,28.988688,9.253969
1,ctc,fleurs,en,100,31.330809,9.067992
2,ctc,mig-burmese-audio-transcription,en,100,24.247013,10.999699
3,ctc,openslr,en,100,28.203514,8.013863
4,ctc,all,ko,300,12.083943,3.117568
5,ctc,fleurs,ko,100,12.166590,2.142719
6,ctc,mig-burmese-audio-transcription,ko,100,10.991260,5.754656
7,ctc,openslr,ko,100,12.616528,2.816803
8,oracle,all,en,300,45.323062,26.076725
9,oracle,fleurs,en,100,52.761507,31.674583


,language,old_ctc,old_oracle,new_ctc,new_oracle,ctc_gain,oracle_change,pass
0,en,26.609848,44.980627,28.988688,45.323062,2.378840,0.342435,True
1,ko,11.715379,22.797998,12.083943,22.581792,0.368564,-0.216206,False


,sample_id,source,language,condition,reference,prediction
0,train_001180,fleurs,en,oracle,Xinhua reported that government investigators ...,State investigators reportedly recovered two B...
1,train_002169,fleurs,en,oracle,A power failure following a routine fire-comma...,A power failure following a routine fire comma...
100,train_006319,mig-burmese-audio-transcription,en,oracle,Calculating that they could make us dance to t...,They calculated that we could secure it by mak...
101,train_006907,mig-burmese-audio-transcription,en,oracle,But in reality...,But in fact...
200,train_005665,openslr,en,oracle,"When we attended university, there was a unive...","When we arrived at university, there was an ar..."
201,train_006056,openslr,en,oracle,I do not want to move to the other side before...,I do not want to go back that way without fini...
300,train_001180,fleurs,en,ctc,Xinhua reported that government investigators ...,The Royal Commission into Abuse of Power revea...
301,train_002169,fleurs,en,ctc,A power failure following a routine fire-comma...,The volcanic activity was triggered by a serie...
400,train_006319,mig-burmese-audio-transcription,en,ctc,Calculating that they could make us dance to t...,They calculated that these were objects of att...
401,train_006907,mig-burmese-audio-transcription,en,ctc,But in reality...,But in fact...


## 채택 기준

- CTC chrF++가 기존보다 언어별 최소 +2
- Oracle chrF++ 하락이 1점 이내
- 두 조건을 모두 만족한 adapter만 최종 채택
- 실패해도 기존 adapter는 덮어쓰지 않았으므로 즉시 복귀 가능
